# Wider residual network + AugMix experiments (v2)

Two new **4,972,345-parameter** models, each trained for **400 epochs**:

1. **wide_main**: the current recipe with stage widths increased to 59 / 118 / 236.
2. **wide_augmix**: the wider model with AugMix and three-view consistency training.

Both use the original 45,000 / 5,000 split, seed 2026. Your existing main development
checkpoint is a third candidate, so a new experiment is not automatically declared better.
No pretrained weights or external images are used. Keep your original submission and checkpoints.

Use an **A100 GPU runtime**. Code and caches live in a new local directory; checkpoints go
to **MyDrive/cifar_resnet_project/runs_v2**. The old `runs` folder is read only.
AugMix processes three views per image, so it is more expensive than the original recipe.
The timing cell measures each recipe separately; your old 18.6-second estimate does not apply.

This notebook first performs development experiments. Full-data fitting is **off by default**
until you inspect their results. An accuracy improvement, especially on Kaggle, is not guaranteed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, sys, subprocess, shutil
import torch

PROJECT = Path('/content/drive/MyDrive/cifar_resnet_project')
RUNS = PROJECT / 'runs_v2'
ORIGINAL = PROJECT / 'runs/main'
DATA_ZIP = PROJECT / 'data.zip'
if not DATA_ZIP.is_file():
    DATA_ZIP = Path('/content/data.zip')  # Or upload data.zip via the left Files panel.
assert DATA_ZIP.is_file(), 'Upload data.zip to Drive or /content/data.zip; leave it zipped.'
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
assert (ORIGINAL / 'best.pt').is_file() and (ORIGINAL / 'final.pt').is_file(), 'Keep the completed original main run in runs/main.'
WORKERS = 4
DETERMINISTIC = False
RUN_ROBUSTNESS = True
RUNS.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Original checkpoints:', ORIGINAL)
print('New experiments:', RUNS)

## Load the v2 code and pin its version

This notebook uses a separate clone, so the original notebook's saved code version does
not hide the new AugMix support. On reconnection, rerun from the beginning. A saved commit
in `runs_v2/code_commit.txt` keeps resumed runs on the same code version. Do not edit configs,
batch size, the deterministic setting, or epoch budget while resuming an experiment.

In [ ]:
LOCAL = Path('/content/cifar_competition_v2')
REPO = LOCAL / 'code'
LOCAL.mkdir(exist_ok=True)
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/mj3664-web/cifar-resnet-colab.git', str(REPO)], check=True)
pin = RUNS / 'code_commit.txt'
if pin.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', pin.read_text().strip()], check=True)
else:
    assert (REPO / 'configs/wide_augmix.json').is_file(), 'The v2 files are missing from this clone.'
    commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
    pin.write_text(commit)
print('Pinned code commit:', pin.read_text())
os.chdir(REPO)

def run_module(module, *args):
    command = [sys.executable, '-u', '-m', module, *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=REPO, check=True)

def train(config, output, full_data=False, stop_after=None):
    args = ['--data', CACHE, '--config', config, '--out', output, '--workers', WORKERS, '--resume']
    if full_data:
        args += ['--full-data']
    if stop_after is not None:
        args += ['--stop-after', stop_after]
    if DETERMINISTIC:
        args += ['--deterministic']
    run_module('src.train', *args)

## Validate the software and prepare data

Tests include exact parameter counts, AugMix consistency-loss gradients, and interruption /
resumption for both training paths. Tiny synthetic fixtures are used only for software tests.
The actual competition training uses only your original archive.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO, check=True)
CACHE = LOCAL / 'cache'
local_zip = LOCAL / 'data.zip'
shutil.copy2(DATA_ZIP, local_zip)
run_module('src.data', '--zip', local_zip, '--out', CACHE)
manifest = json.loads((CACHE / 'manifest.json').read_text())
assert manifest['train_count'] == 50000 and manifest['test_count'] == 2000
original_meta = json.loads((ORIGINAL / 'metadata.json').read_text())
assert original_meta['archive_sha256'] == manifest['archive_sha256'], 'Original run used a different archive.'
for recipe in ['wide_main', 'wide_augmix']:
    c = json.loads((REPO / 'configs' / (recipe + '.json')).read_text())
    assert c['split_seed'] == original_meta['config']['split_seed']
    assert c['val_fraction'] == original_meta['config']['val_fraction']
print('Dataset and validation split match the original run.')

## Measure each recipe's time

These are the first three epochs of each real 400-epoch schedule, not throwaway training.
The next cell resumes them. AugMix uses **128 source images and 384 total views per batch**;
it still has only 4,972,345 model parameters. If you hit an out-of-memory error, use a fresh
experiment directory with a smaller batch/config; do not change a saved run in place.

In [ ]:
RECIPES = ['wide_main', 'wide_augmix']
for recipe in RECIPES:
    directory = RUNS / recipe
    train(REPO / 'configs' / (recipe + '.json'), directory, stop_after=3)
    rows = [json.loads(s) for s in (directory / 'history.jsonl').read_text().splitlines()]
    recent = rows[-min(3, len(rows)):]
    seconds = sum(r['seconds'] for r in recent) / len(recent)
    print(f'{recipe}: {seconds:.1f} seconds/epoch; about {seconds * 400 / 3600:.1f} hours for 400 epochs.')

## Complete both development runs

Each run starts from random weights and uses the same data split. `wide_main` changes only
channel width relative to the old main recipe. `wide_augmix` replaces RandAugment and
MixUp/CutMix with AugMix plus a Jensen–Shannon consistency loss (weight 12). The optimizer,
batch size, 400-epoch schedule, seed, EMA, and inference settings remain the same.

AugMix loss = classification loss on the base crop + 12 × consistency loss across the
base crop and two independently mixed views. There is no late mixing-off transition in this
recipe. Its total loss is not directly comparable with the original mixed-label loss.

In [ ]:
for recipe in RECIPES:
    train(REPO / 'configs' / (recipe + '.json'), RUNS / recipe)

## Compare completed runs and select using clean validation accuracy

The old main model participates in selection. A tiny difference can be seed noise: a
0.1 percentage-point change is only five of the 5,000 held-out examples. Consider confirming
close results with another seed before committing to a new full-data fit.

In [ ]:
CANDIDATES = [ORIGINAL] + [RUNS / r for r in RECIPES]
SELECTED_CONFIG = RUNS / 'selected.json'
run_module('src.select', '--runs', *CANDIDATES, '--out', SELECTED_CONFIG)
selection = json.loads(SELECTED_CONFIG.with_suffix('.selection.json').read_text())
for candidate in selection['candidates']:
    print(f"{Path(candidate['run']).name}: {100 * candidate['accuracy']:.2f}%")
print('Selected:', selection['selected_run'], selection['weights'])

## Fixed robustness checks on held-out training images

Optional diagnostic: clean images, brightness ×0.7, contrast ×0.6, Gaussian blur radius 1,
and Gaussian noise sigma 0.05 (in 0–1 pixel units). These transformations and their strength
are fixed in advance. All models see the same held-out images and deterministic noise.

This small custom stress test is **not CIFAR-10-C and does not estimate Kaggle accuracy**.
It never reads hidden test images or labels. The automatic selector above still uses clean
validation accuracy; inspect these supplementary results before deciding to do more training.

In [ ]:
if RUN_ROBUSTNESS:
    run_module('src.robustness', '--data', CACHE,
               '--checkpoints', *[c / 'best.pt' for c in CANDIDATES],
               '--out', RUNS / 'robustness.json', '--workers', WORKERS)
    report = json.loads((RUNS / 'robustness.json').read_text())
    for result in report['results']:
        print(Path(result['checkpoint']).parent.name,
              {k: round(100*v, 2) for k, v in result['accuracy'].items()},
              'shifted mean:', round(100 * result['shifted_mean_accuracy'], 2))
else:
    print('Robustness checks skipped.')

## Optional: final fit and a separate submission

After reviewing the comparison, set **RUN_FINAL_FIT = True** below to train the selected
new recipe on all 50,000 images for 400 epochs. This is additional GPU work, with no held-out
accuracy from that full-data model. If the original main model wins, reuse its existing final
checkpoint instead of retraining. The new CSV is saved under `submissions_v2`; the original
`submission.csv` and all original checkpoints are preserved.

No submission is sent automatically to Kaggle. Any official score remains unknown until
you upload the generated CSV through the competition's submission process.

In [ ]:
RUN_FINAL_FIT = False  # Change to True only after inspecting the development results above.
if RUN_FINAL_FIT:
    chosen_run = Path(selection['selected_run'])
    if chosen_run == ORIGINAL:
        FINAL_CHECKPOINT = PROJECT / 'runs/final/final.pt'
        assert FINAL_CHECKPOINT.is_file(), 'Locate the final checkpoint from your original completed run.'
        tag = 'original_main'
    else:
        tag = chosen_run.name
        FINAL_RUN = RUNS / ('final_' + tag)
        train(SELECTED_CONFIG, FINAL_RUN, full_data=True)
        FINAL_CHECKPOINT = FINAL_RUN / 'final.pt'
    SUBMISSION = PROJECT / 'submissions_v2' / tag / 'submission.csv'
    run_module('src.predict', '--data', CACHE, '--checkpoint', FINAL_CHECKPOINT,
               '--out', SUBMISSION, '--workers', WORKERS)
    record = {'selection': selection, 'code_commit': pin.read_text().strip(),
              'submission': json.loads(SUBMISSION.with_suffix('.metadata.json').read_text()),
              'official_kaggle_score': None}
    (SUBMISSION.parent / 'reproduction_record.json').write_text(json.dumps(record, indent=2))
    from google.colab import files
    files.download(str(SUBMISSION))
else:
    print('Development comparison complete. Final fitting is off; no new submission was generated.')